# PlazaPD · Data

Saved snapshot · DuckDB SQL + Pandas tables · Run All. [DuckDB notebooks](https://duckdb.org/docs/current/guides/python/jupyter).

In [1]:
import json
import sys
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/plazapd/mappedin.py").is_file()
             and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or nbs/plazapd.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.plazapd.mappedin import load_tables

DATA_DIR = ROOT / "data/plazapd"
tables = load_tables(DATA_DIR, format="parquet")
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
assert tables["entities"].entity_id.is_unique
assert tables["relationships"][["src", "dst"]].isin(set(tables["entities"].entity_id)).all().all()
assert tables["distances"].loc[tables["distances"].status.eq("unreachable"), "distance_m"].isna().all()
pd.options.display.max_colwidth = 50
pd.options.display.max_rows = 25

db = duckdb.connect(":memory:")
for name, frame in tables.items():
    db.register(name, frame)


def sql(query, parameters=None):
    return db.execute(query, parameters).df()

## Inventory

Stores and amenities are subsets of locations; locations and structural collections are subsets of entities.

In [2]:
display(pd.Series({k: manifest.get(k) for k in ["venue_name", "collected_at", "map_updated_at"]}, name="snapshot").to_frame())
inventory = pd.DataFrame([{"table": name, "rows": len(t), "columns": len(t.columns)} for name, t in tables.items()])
display(inventory)
display(sql("SELECT collection, type, count(*) AS entities FROM entities GROUP BY ALL ORDER BY entities DESC"))
display(sql("SELECT relation, count(*) AS edges FROM relationships GROUP BY ALL ORDER BY edges DESC"))
display(sql("SELECT entity_id, name FROM venues"))

,snapshot
venue_name,Plaza San Miguel
collected_at,2026-11-14T23:07:35.165Z
map_updated_at,2026-11-12T23:23:05.353Z


,table,rows,columns
0,entities,3954,14
1,relationships,14047,3
2,distances,186192,4
3,locations,448,14
4,categories,14,14
5,venues,1,14
6,floors,9,14
7,buildings,1,14
8,spaces,1133,14
9,connections,73,14


,collection,type,entities
0,nodes,node,2125
1,spaces,space,1133
2,locations,enterprise-location,448
3,points_of_interest,point-of-interest,150
4,connections,connection,73
5,categories,enterprise-category,14
6,floors,floor,9
7,venues,enterprise-venue,1
8,buildings,floor-stack,1


,relation,edges
0,neighbor,5345
1,floor,3408
2,spaces,1566
3,locations,1373
4,nodes,832
5,space,770
6,categories,351
7,floors,252
8,points,150


,entity_id,name
0,enterprise-venue:VENUE-29DTWCLZ,Plaza San Miguel


In [3]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW location_spaces AS
SELECT DISTINCT r.src AS location_id, r.dst AS space_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN spaces s ON s.entity_id = r.dst WHERE r.relation = 'spaces';

CREATE OR REPLACE TEMP VIEW space_floors AS
SELECT DISTINCT r.src AS space_id, r.dst AS floor_entity_id
FROM relationships r JOIN spaces s ON s.entity_id = r.src
JOIN floors f ON f.entity_id = r.dst WHERE r.relation = 'floor';

CREATE OR REPLACE TEMP VIEW location_floors AS
SELECT DISTINCT l.location_id, f.floor_entity_id
FROM location_spaces l JOIN space_floors f USING (space_id);

CREATE OR REPLACE TEMP VIEW location_category_links AS
SELECT DISTINCT r.src AS location_id, r.dst AS category_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN categories c ON c.entity_id = r.dst WHERE r.relation = 'categories';

CREATE OR REPLACE TEMP VIEW node_catalog AS
SELECT e.entity_id, e.id, coalesce(nullif(trim(e.name), ''), e.entity_id) AS label,
       CASE WHEN s.entity_id IS NOT NULL THEN 'store'
            WHEN a.entity_id IS NOT NULL THEN 'amenity'
            WHEN e.collection = 'spaces' THEN 'space'
            WHEN e.collection = 'floors' THEN 'floor'
            WHEN e.collection = 'categories' THEN 'category'
            WHEN e.collection = 'nodes' THEN 'navigation'
            ELSE e.type END AS kind,
       e.hidden, e.navigable, e.amenity
FROM entities e LEFT JOIN stores s USING (entity_id) LEFT JOIN amenities a USING (entity_id);
""")

## Stores and amenities × floor

Distinct locations per floor; multi-floor locations contribute to each linked floor.

In [4]:
floor_counts = sql("""
SELECT f.entity_id, f.name AS floor, f.elevation,
       count(DISTINCT l.location_id) FILTER (WHERE n.kind = 'store') AS stores,
       count(DISTINCT l.location_id) FILTER (WHERE n.kind = 'amenity') AS amenities
FROM floors f LEFT JOIN location_floors l ON l.floor_entity_id = f.entity_id
LEFT JOIN node_catalog n ON n.entity_id = l.location_id
GROUP BY ALL ORDER BY f.elevation, f.entity_id
""")
display(floor_counts)

,entity_id,floor,elevation,stores,amenities
0,floor:f_6995e7d23c9e283a4ccb95bd,Sotano 4,-4.0,0,4
1,floor:f_6995e893ec1c8c8fc2645e2c,Sotano 3,-3.0,0,4
2,floor:f_6995e6eca61b9a6db33606a6,Sotano 2,-2.0,0,4
3,floor:f_6995e6a701f04884410ff878,Sotano 1,-1.0,8,10
4,floor:f_6995ac1a984770e45e357535,Piso 1,0.0,162,7
5,floor:f_6995db2cf4b88c241bc8afbc,Piso 2,1.0,113,3
6,floor:f_6995e0e34d98e7eb09c7df16,Piso 3,2.0,53,4
7,floor:f_6996ef8632895508b8d0967e,Piso 4,3.0,7,3
8,floor:f_69986a6add160b20bdaa328f,Piso 5,4.0,2,0


## Categories × floor

Counts use distinct stores; multi-category and multi-floor memberships overlap.

In [5]:
category_floor = sql("""
SELECT c.name AS category, f.name AS floor, count(DISTINCT s.entity_id) AS stores
FROM stores s JOIN location_category_links lc ON lc.location_id = s.entity_id
JOIN categories c ON c.entity_id = lc.category_id
JOIN location_floors lf ON lf.location_id = s.entity_id
JOIN floors f ON f.entity_id = lf.floor_entity_id
GROUP BY ALL ORDER BY category, floor
""")
floor_order = tables["floors"].sort_values(["elevation", "entity_id"]).name.tolist()
display(category_floor.pivot(index="category", columns="floor", values="stores").reindex(
    index=sorted(tables["categories"].name), columns=floor_order).fillna(0).astype(int))
display(sql("""
SELECT c.name AS category, count(DISTINCT s.entity_id) AS stores
FROM categories c LEFT JOIN location_category_links lc ON lc.category_id = c.entity_id
LEFT JOIN stores s ON s.entity_id = lc.location_id GROUP BY ALL ORDER BY stores DESC, category
"""))

floor,Sotano 4,Sotano 3,Sotano 2,Sotano 1,Piso 1,Piso 2,Piso 3,Piso 4,Piso 5
category,,,,,,,,,
Accesorios y Joyería,0,0,0,1,18,11,0,0,0
Belleza y Salud,0,0,0,3,29,15,1,0,0
Calzado y Artículos de Cuero,0,0,0,0,19,5,0,0,0
Deportes y Outdoor,0,0,0,0,12,5,0,2,0
Entretenimiento y Cultura,0,0,0,0,6,5,11,2,2
Gastronomía,0,0,0,0,34,21,16,1,0
Hogar y Decoración,0,0,0,2,4,11,2,0,0
Moda,0,0,0,1,20,20,0,0,0
Niños,0,0,0,0,2,3,4,0,0


,category,stores
0,Gastronomía,76
1,Belleza y Salud,49
2,Moda,41
3,Accesorios y Joyería,31
4,Servicios y Banca,29
5,Calzado y Artículos de Cuero,24
6,Entretenimiento y Cultura,24
7,Tecnología,22
8,Deportes y Outdoor,19
9,Hogar y Decoración,19


## Stores with multiple spaces or floors

In [6]:
store_spatial = sql("""
SELECT s.entity_id, s.name, count(DISTINCT ls.space_id) AS spaces,
       count(DISTINCT lf.floor_entity_id) AS floors,
       string_agg(DISTINCT f.name, ' | ' ORDER BY f.name) AS floor_names
FROM stores s LEFT JOIN location_spaces ls ON ls.location_id = s.entity_id
LEFT JOIN location_floors lf ON lf.location_id = s.entity_id
LEFT JOIN floors f ON f.entity_id = lf.floor_entity_id
GROUP BY ALL ORDER BY floors DESC, spaces DESC, s.name, s.entity_id
""")
display(store_spatial.loc[store_spatial.spaces.gt(1) | store_spatial.floors.gt(1)])
display(store_spatial.loc[store_spatial.spaces.eq(0), ["entity_id", "name", "spaces", "floors"]])

,entity_id,name,spaces,floors,floor_names
0,enterprise-location:el_69b0548a2c71c6f75252b8e6,FALABELLA,4,4,Piso 1 | Piso 2 | Piso 3 | Piso 4
1,enterprise-location:el_69b0548a2c71c6f75252b963,RIPLEY,4,4,Piso 1 | Piso 2 | Piso 3 | Piso 4
2,enterprise-location:el_6a207bbb151b4d0128140e2d,CINE PLANET PRIME,3,3,Piso 3 | Piso 4 | Piso 5
3,enterprise-location:el_69b0548a2c71c6f75252b8e1,BITEL,2,2,Piso 1 | Piso 3
4,enterprise-location:el_69b0548a2c71c6f75252b9ae,PLAZA VEA,2,2,Piso 1 | Piso 2
5,enterprise-location:el_69b0548a2c71c6f75252b964,WONG,2,2,Piso 1 | Piso 2
6,enterprise-location:el_69b0548a2c71c6f75252b9e6,DON SALAZAR,2,1,Piso 2


,entity_id,name,spaces,floors
334,enterprise-location:el_69b0548a2c71c6f75252b90c,BANCO DE NACION - CAJERO,0,0
335,enterprise-location:el_69b0548a2c71c6f75252b90e,BCP - CAJERO,0,0
336,enterprise-location:el_6aee6e2e85a94c0410eeedcd,BOBOCHA,0,0
337,enterprise-location:el_69b0548a2c71c6f75252b9bc,DELICASS,0,0
338,enterprise-location:el_69b0548a2c71c6f75252b9f1,ESTRELLA DORADA,0,0
339,enterprise-location:el_69b0548a2c71c6f75252b9ee,EXPENDOMAX,0,0
340,enterprise-location:el_69b0548a2c71c6f75252b9fa,FIT SERVICES,0,0
341,enterprise-location:el_69b0548a2c71c6f75252b995,LA GALLETERIA,0,0
342,enterprise-location:el_6aeba7d1dfdd4a4e87797ef4,MINAJI,0,0
343,enterprise-location:el_69b0548a2c71c6f75252b9dc,NATURALE,0,0


## Missing links and navigation coverage

In [7]:
link_coverage = sql("""
SELECT n.kind, count(*) AS locations,
 count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM location_spaces x WHERE x.location_id=n.entity_id)) AS no_space,
 count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM location_floors x WHERE x.location_id=n.entity_id)) AS no_floor,
 count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM location_category_links x WHERE x.location_id=n.entity_id)) AS no_category,
 count(*) FILTER (WHERE n.hidden) AS hidden,
 count(*) FILTER (WHERE n.navigable) AS navigable,
 count(*) FILTER (WHERE n.navigable IS NULL) AS navigation_unknown
FROM node_catalog n WHERE kind IN ('store','amenity') GROUP BY kind ORDER BY kind
""")
display(link_coverage)
display(sql("SELECT hidden, navigable, count(*) AS stores FROM stores GROUP BY ALL ORDER BY ALL"))

,kind,locations,no_space,no_floor,no_category,hidden,navigable,navigation_unknown
0,amenity,101,86,86,101,6,97,0
1,store,347,13,13,2,0,335,0


,hidden,navigable,stores
0,False,False,12
1,False,True,335


## Directed walking distances · metres

Missing route rows and explicit unreachable pairs are counted separately; unreachable distances remain null.

In [8]:
routing_coverage = sql("""
SELECT count(*) AS navigable_locations, count(*) * (count(*) - 1) AS expected_pairs,
       (SELECT count(*) FROM distances) AS stored_pairs,
       count(*) * (count(*) - 1) - (SELECT count(*) FROM distances) AS missing_pairs
FROM locations WHERE navigable
""")
display(routing_coverage)
display(sql("SELECT status, count(*) AS pairs, count(distance_m) AS measured FROM distances GROUP BY status ORDER BY status"))
display(sql("""
SELECT min(distance_m) AS min_m, quantile_cont(distance_m,0.25) AS p25_m,
 quantile_cont(distance_m,0.5) AS median_m, quantile_cont(distance_m,0.75) AS p75_m,
 quantile_cont(distance_m,0.9) AS p90_m, max(distance_m) AS max_m
FROM distances WHERE status='reachable'
""").round(2))

,navigable_locations,expected_pairs,stored_pairs,missing_pairs
0,432,186192,186192,0


,status,pairs,measured
0,reachable,185330,185330
1,unreachable,862,0


,min_m,p25_m,median_m,p75_m,p90_m,max_m
0,0.0,134.48,209.77,297.87,369.11,589.07


## Nearest reachable amenity per store

Coverage denominators are navigable stores; zero metres is a measured result.

In [9]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW nearest_amenity AS
SELECT s.entity_id, s.name, s.navigable, a.entity_id AS amenity_id,
       a.name AS amenity_name, a.amenity AS amenity_type, d.distance_m
FROM stores s LEFT JOIN distances d ON d.from_id=s.id AND d.status='reachable'
 AND d.to_id IN (SELECT id FROM amenities)
LEFT JOIN amenities a ON a.id=d.to_id
QUALIFY row_number() OVER (PARTITION BY s.entity_id ORDER BY d.distance_m NULLS LAST, a.entity_id)=1
""")
amenity_coverage = sql("""
SELECT count(*) AS stores, count(*) FILTER (WHERE navigable) AS eligible_stores,
 count(distance_m) AS with_reachable_amenity,
 count(*) FILTER (WHERE navigable AND distance_m IS NULL) AS eligible_without_reachable_amenity,
 quantile_cont(distance_m,0.5) AS median_nearest_m, max(distance_m) AS max_nearest_m
FROM nearest_amenity
""")
display(amenity_coverage.round(2))
nearest_distance_summary = sql("""
SELECT min(distance_m) AS min_m, quantile_cont(distance_m,0.25) AS p25_m,
 quantile_cont(distance_m,0.5) AS median_m, quantile_cont(distance_m,0.75) AS p75_m,
 quantile_cont(distance_m,0.9) AS p90_m, max(distance_m) AS max_m
FROM nearest_amenity WHERE distance_m IS NOT NULL
""")
display(nearest_distance_summary.round(2))
display(sql("SELECT name, amenity_name, amenity_type, round(distance_m,2) AS distance_m FROM nearest_amenity WHERE distance_m IS NOT NULL ORDER BY distance_m DESC, entity_id LIMIT 10"))
display(sql("""
SELECT n.name, n.navigable,
 CASE WHEN NOT coalesce(n.navigable,false) THEN 'Not a routing target'
      WHEN EXISTS (SELECT 1 FROM distances d JOIN stores s ON s.id=d.from_id WHERE s.entity_id=n.entity_id)
      THEN 'Route rows exist; no reachable amenity' ELSE 'No route rows' END AS route_coverage
FROM nearest_amenity n WHERE distance_m IS NULL ORDER BY n.name, n.entity_id
"""))

,stores,eligible_stores,with_reachable_amenity,eligible_without_reachable_amenity,median_nearest_m,max_nearest_m
0,347,335,334,1,42.45,124.33


,min_m,p25_m,median_m,p75_m,p90_m,max_m
0,0.0,26.89,42.45,59.42,81.11,124.33


,name,amenity_name,amenity_type,distance_m
0,iSHOP,Descanso accesible,accessible-washroom,124.33
1,NIKE,Baño de mujeres,women-s-washroom,113.75
2,STARBUCKS COFFEE,Baño de mujeres,women-s-washroom,112.80
3,CONEY PARK,Entrada,entrance,104.53
4,KIDS MADE HERE,Baño de mujeres,women-s-washroom,103.92
5,HONOR,Descanso accesible,accessible-washroom,102.39
6,PIONIER,Baño de mujeres,women-s-washroom,96.76
7,COOLBOX 2,Baño de hombres,men-s-washroom,95.62
8,LEONISA,Baño de mujeres,women-s-washroom,95.19
9,PARDO'S CHICKEN,Baño de mujeres,women-s-washroom,94.69


,name,navigable,route_coverage
0,ADIDAS,True,Route rows exist; no reachable amenity
1,BANCO DE NACION - CAJERO,False,Not a routing target
2,BCP - CAJERO,False,Not a routing target
3,BOBOCHA,False,Not a routing target
4,ESTRELLA DORADA,False,Not a routing target
5,EXPENDOMAX,False,Not a routing target
6,FIT SERVICES,False,Not a routing target
7,LA GALLETERIA,False,Not a routing target
8,MINAJI,False,Not a routing target
9,NATURALE,False,Not a routing target


## Nearest amenities × type

Each type uses all navigable stores as its denominator.

In [10]:
amenity_type_coverage = sql("""
WITH types AS (SELECT DISTINCT coalesce(nullif(amenity,''),'Unspecified') AS amenity_type FROM amenities),
eligibility AS (SELECT count(*) AS eligible_stores FROM stores WHERE navigable),
nearest AS (
 SELECT d.from_id, coalesce(nullif(a.amenity,''),'Unspecified') AS amenity_type, min(d.distance_m) AS nearest_m
 FROM distances d JOIN stores s ON s.id=d.from_id AND s.navigable
 JOIN amenities a ON a.id=d.to_id WHERE d.status='reachable' GROUP BY ALL
)
SELECT t.amenity_type, e.eligible_stores,
 count(n.from_id) AS with_reachable_amenity, eligible_stores-count(n.from_id) AS without_reachable_amenity,
 round(100.0*count(n.from_id)/nullif(eligible_stores,0),1) AS coverage_pct,
 round(quantile_cont(n.nearest_m,0.5),2) AS median_nearest_m, round(max(n.nearest_m),2) AS max_nearest_m
FROM types t CROSS JOIN eligibility e LEFT JOIN nearest n USING (amenity_type)
GROUP BY t.amenity_type, e.eligible_stores ORDER BY coverage_pct, t.amenity_type
""")
display(amenity_type_coverage)

,amenity_type,eligible_stores,with_reachable_amenity,without_reachable_amenity,coverage_pct,median_nearest_m,max_nearest_m
0,gift-card,335,0,335,0.0,NaN,NaN
1,Unspecified,335,334,1,99.7,88.91,198.79
2,accessible-washroom,335,334,1,99.7,99.88,363.91
3,atm,335,334,1,99.7,208.56,421.70
4,bike-rack,335,334,1,99.7,303.10,557.98
5,entrance,335,334,1,99.7,93.96,351.65
6,family-washroom,335,334,1,99.7,194.34,435.52
7,food-court,335,334,1,99.7,267.53,488.27
8,interactive-display,335,334,1,99.7,95.08,301.18
9,men-s-washroom,335,334,1,99.7,90.06,296.86


## Forward versus reverse routes

Each unordered pair appears once; distance differences use a 0.01-metre tolerance.

In [11]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW route_comparison AS
SELECT a.from_id, a.to_id, a.status AS forward_status, b.status AS reverse_status,
       a.distance_m AS forward_m, b.distance_m AS reverse_m,
       abs(a.distance_m-b.distance_m) AS difference_m
FROM distances a JOIN distances b ON a.from_id=b.to_id AND a.to_id=b.from_id
WHERE a.from_id<a.to_id
""")
route_asymmetry = sql("""
SELECT count(*) AS unordered_pairs,
 count(*) FILTER (WHERE forward_status<>reverse_status) AS unequal_reachability,
 count(*) FILTER (WHERE forward_status='reachable' AND reverse_status='reachable') AS both_reachable,
 count(*) FILTER (WHERE difference_m>0.01) AS distance_diff_over_1cm
FROM route_comparison
""")
display(route_asymmetry)
display(sql("""
SELECT f.name AS source, t.name AS destination, round(r.forward_m,2) AS forward_m,
 round(r.reverse_m,2) AS reverse_m, round(r.difference_m,2) AS difference_m
FROM route_comparison r JOIN locations f ON f.id=r.from_id JOIN locations t ON t.id=r.to_id
WHERE r.difference_m>0.01 ORDER BY r.difference_m DESC, r.from_id, r.to_id LIMIT 10
"""))

,unordered_pairs,unequal_reachability,both_reachable,distance_diff_over_1cm
0,93096,0,92665,12671


,source,destination,forward_m,reverse_m,difference_m
0,FALABELLA,DOLLAR CITY,65.19,289.77,224.58
1,LA BISTECCA PASTA GRILL,WAYU,236.86,38.48,198.39
2,LA BISTECCA PASTA GRILL,AZALEIA,293.94,95.55,198.39
3,LA BISTECCA PASTA GRILL,SWAROVSKI,242.28,43.89,198.39
4,LA BISTECCA PASTA GRILL,CAFFARENA,299.82,101.44,198.39
5,LA BISTECCA PASTA GRILL,VICTORIA'S SECRET,281.79,83.41,198.39
6,CREPIER,LA BISTECCA PASTA GRILL,103.15,301.54,198.39
7,LA BISTECCA PASTA GRILL,LACOSTE,265.83,67.45,198.39
8,LA BISTECCA PASTA GRILL,PIERRE CARDIN,275.34,76.96,198.39
9,LA BISTECCA PASTA GRILL,MAISON ARABIA,303.82,105.43,198.39
